# 02 — Kaggle 提出用 submission.tar.gz 作成

**`.py` 単体は提出不可。** 以下 3 つを `submission.tar.gz` に同梱してください。

- `main.py`（agents/*.py をリネーム）
- `cg/`（cg-lib Input 内の cg フォルダ）
- `deck.csv`（デッキ用 Input）

## 前提

- Input: **cg-lib**
- Input: 対象デッキの **deck.csv** を含むデータセット
- このリポジトリを Add Data するか、`agents/` 配下の py を Working に配置

In [ ]:
AGENT_KEY = "mega-abomasnow-ex"  # mega-abomasnow-ex | dragapult-ex | iono-s | mega-lucario-ex

AGENT_FILES = {
    "mega-abomasnow-ex": "mega_abomasnow_ex.py",
    "dragapult-ex": "dragapult_ex.py",
    "iono-s": "iono_s.py",
    "mega-lucario-ex": "mega_lucario_ex.py",
}

In [ ]:
import glob
import os
import tarfile
from pathlib import Path

agent_name = AGENT_FILES[AGENT_KEY]
candidates = [
    Path(f"/kaggle/input/kaggle/Pokemon/agents/{agent_name}"),
    Path(f"../agents/{agent_name}"),
    Path(f"agents/{agent_name}"),
]
agent_path = next((p for p in candidates if p.is_file()), None)
if agent_path is None:
    raise FileNotFoundError(f"Agent not found: {agent_name}. Checked: {candidates}")

cg_path = glob.glob("/kaggle/input/**/cg-lib/cg", recursive=True)[0]
deck_path = glob.glob("/kaggle/input/**/deck.csv", recursive=True)[0]

with open(agent_path, "r", encoding="utf-8") as src, open("main.py", "w", encoding="utf-8") as dst:
    dst.write(src.read())

with tarfile.open("submission.tar.gz", "w:gz") as tar:
    tar.add("main.py", arcname="main.py")
    tar.add(cg_path, arcname="cg")
    tar.add(deck_path, arcname="deck.csv")

with tarfile.open("submission.tar.gz", "r:gz") as tar:
    names = tar.getnames()
print("Created submission.tar.gz")
print("members:", names[:10], "..." if len(names) > 10 else "")
assert "main.py" in names
assert "deck.csv" in names
assert any(n.startswith("cg/") or n == "cg" for n in names), names

os.remove("main.py")
print("OK — submit submission.tar.gz to the competition")